# WikiText headline protocol — full pipeline (fitted templates + controls)

Runs the paper's **full headline protocol on WikiText** (natural text) at Qwen3-0.6B: fit rich
templates for all 448 heads, measure solo costs (natural + repeated-random), build MLP tables,
then the **position-shuffle control pair** (real code + heal vs shuffled code + heal, 20 epochs,
fp32 — protocol-matched to the fiction headline) and the **intact-heal offset**.

MLP layers are fixed to the middle band [9–14]: the solo scan on WikiText selects early layers
whose joint cost explodes (+3.49 vs +0.36) — solo mispredicts joint.

`Runtime → Change runtime type → GPU`, then `Run all`. Total ~1–2 h on a T4.

Repo: https://github.com/keithagroves/how-much-transformer-is-code

In [ ]:
!pip install -q transformers datasets accelerate
import torch; print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU — set Runtime to GPU')


In [ ]:
#@title Step 1 · Fetch pipeline scripts
BASE = 'https://raw.githubusercontent.com/keithagroves/how-much-transformer-is-code/main/src'
for f in ['replace_all.py','replace_rich.py','rnd_solo.py','mlp_prosthesis.py','heal_shuffle.py','heal_intact_baseline.py','decomp_diag.py']:
    !wget -q -O {f} {BASE}/{f}
print('scripts ready')


In [ ]:
#@title Step 2 · Build the WikiText corpus (1.4M chars, same construction as the paper)
from datasets import load_dataset
ds = load_dataset('Salesforce/wikitext', 'wikitext-103-raw-v1', split='train', streaming=True)
buf, n = [], 0
for row in ds:
    t = row['text'].strip()
    if len(t) > 80:
        buf.append(t); n += len(t)
    if n >= 1_400_000: break
open('wikitext_corpus.txt','w').write('\n'.join(buf))
print('corpus chars:', n)


In [ ]:
#@title Step 3 · Run the chain (fits → solo costs → MLP tables → shuffle pair → offset)
import os
os.environ['SUB_CORPUS'] = 'wikitext_corpus.txt'
os.environ['SUB_MLPS'] = '9,10,11,12,13,14'
!python -W ignore replace_rich.py 2>&1 | tail -20
print('=== DONE_RICH ===')
!python -W ignore rnd_solo.py 2>&1 | tail -3
print('=== DONE_RND ===')
!python -W ignore mlp_prosthesis.py 2>&1 | tail -8
print('=== DONE_MLP ===')
!python -W ignore heal_shuffle.py
print('=== DONE_SHUFFLE ===')
!python -W ignore heal_intact_baseline.py 2>&1 | tail -5
print('=== DONE_ALL ===')


### Reading the output
- **real code** healed damage + the offset (≈ +0.27) = the WikiText **fair cost**, directly
  comparable to the fiction-corpus fair +0.88 (same protocol, different corpus).
- **shuffled code** must heal far worse than real code, or the recovery is the heal, not the code.
- Unhealed reference points measured locally: heads-only +1.69, +MLPs(middle-6) +1.86.
